In [1]:
try:
    import gradio
except ModuleNotFoundError:
    !pip install -U gradio
    import gradio

In [4]:
import gradio as gr
import transformers
import datasets
import torchvision
import math
import torch
import numpy as np
print("[INFO] torch version available ",torch.__version__)
print("[INFO] transformers version available ",transformers.__version__)
print("[INFO] gradio version available ",gradio.__version__)
print("[INFO] datasets version available ",datasets.__version__)

[INFO] torch version available  2.10.0+cpu
[INFO] transformers version available  5.0.0
[INFO] gradio version available  5.50.0
[INFO] datasets version available  4.0.0


## Building and gradio interface for ***Trashify***

In [ ]:
# demos/
# └── trashify_object_detector/
#     ├── app.py
#     ├── README.md
#     ├── requirements.txt
#     └── trashify_examples/
#         ├── trashify_example_1.jpeg
#         ├── trashify_example_2.jpeg
#         └── trashify_example_3.jpeg

## the structure to upload on hugging face

In [15]:
from pathlib import Path

# Setup path to trashify demo folder (we'll store all of our demo requirements in here)
demo_path = Path("/content/demos/trashify_object_detector")
# Create the directory
demo_path.mkdir(parents=True, exist_ok=True)

In [8]:
## importing model
import gradio as gr
import torch
from PIL import Image, ImageDraw, ImageFont # could also use torch utilities for drawing

from transformers import AutoImageProcessor
from transformers import AutoModelForObjectDetection

In [9]:
model_path_saved = "RahulKate-173/rt_detrv2_finetuned_trashify_box_detector_v1"
model = AutoModelForObjectDetection.from_pretrained(model_path_saved)
image_processor = AutoImageProcessor.from_pretrained(model_path_saved)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--RahulKate-173--rt_detrv2_finetuned_trashify_box_detector_v1/snapshots/7151b50d911413add1126905a5c19d18cf7624c3/config.json
Model config RTDetrResNetConfig {
  "depths": [
    3,
    4,
    6,
    3
  ],
  "downsample_in_bottleneck": false,
  "downsample_in_first_stage": false,
  "dtype": "float32",
  "embedding_size": 64,
  "hidden_act": "relu",
  "hidden_sizes": [
    256,
    512,
    1024,
    2048
  ],
  "layer_type": "bottleneck",
  "model_type": "rt_detr_resnet",
  "num_channels": 3,
  "out_features": [
    "stage2",
    "stage3",
    "stage4"
  ],
  "out_indices": [
    2,
    3,
    4
  ],
  "stage_names": [
    "stem",
    "stage1",
    "stage2",
    "stage3",
    "stage4"
  ],
  "transformers_version": "5.0.0"
}

`backbone_config` and `backbone` are `None`. Initializing the config with the default `RTDetrV2-ResNet` backbone.
Model config RTDetrV2Config {
  "activation_dropout": 0.0,
  "a

model.safetensors:   0%|          | 0.00/169M [00:00<?, ?B/s]

loading weights file model.safetensors from cache at /root/.cache/huggingface/hub/models--RahulKate-173--rt_detrv2_finetuned_trashify_box_detector_v1/snapshots/7151b50d911413add1126905a5c19d18cf7624c3/model.safetensors
Will use dtype=torch.float32 as defined in model's config object


Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/513 [00:00<?, ?B/s]

loading configuration file preprocessor_config.json from cache at /root/.cache/huggingface/hub/models--RahulKate-173--rt_detrv2_finetuned_trashify_box_detector_v1/snapshots/7151b50d911413add1126905a5c19d18cf7624c3/preprocessor_config.json
loading configuration file preprocessor_config.json from cache at /root/.cache/huggingface/hub/models--RahulKate-173--rt_detrv2_finetuned_trashify_box_detector_v1/snapshots/7151b50d911413add1126905a5c19d18cf7624c3/preprocessor_config.json
Image processor RTDetrImageProcessorFast {
  "data_format": "channels_first",
  "do_convert_annotations": true,
  "do_normalize": false,
  "do_pad": true,
  "do_rescale": true,
  "do_resize": true,
  "format": "coco_detection",
  "image_mean": [
    0.485,
    0.456,
    0.406
  ],
  "image_processor_type": "RTDetrImageProcessorFast",
  "image_std": [
    0.229,
    0.224,
    0.225
  ],
  "resample": 2,
  "rescale_factor": 0.00392156862745098,
  "return_segmentation_masks": true,
  "size": {
    "longest_edge": 640,

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

In [11]:
# Get the id2label dictionary from the model
id2label = model.config.id2label

# Set up a colour dictionary for plotting boxes with different colours
color_dict = {
    "bin": "green",
    "trash": "blue",
    "hand": "purple",
    "trash_arm": "yellow",
    "not_trash": "red",
    "not_bin": "red",
    "not_hand": "red",
}

In [12]:
def predict_image(image,conf_threshold):
  model.eval()
  with torch.no_grad:
    inputs = image_processor.preprocess(images=[image],return_tensors="pt")
    model_outputs  = model(**inputs.to(device))
    target_size = torch.tensor([[image.shape[0],image.shape[1]]]) # height and width in batch
    result = image_processor.post_process_object_detection(
        images = model_outputs,
        threshold = conf_threshold,
        target_sizes = target_size
    )

    for key,value in result.items():
      try:
        result[key] = value.items().cpu()
      except:
        result[key] = value.cpu()

    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default(size=20)
    detected_class_name_text_labels = []

    for box,score,label in  zip(result["boxes"],result["scores"],result["labels"]):
      x,y,x2,y2 = tuple(box.tolist())
      label_name = id2label[label.item()]
      targ_color = color_dict[label_name]
      detected_class_name_text_labels.append(label_name)

      draw.rectangle(xy=(x, y, x2, y2),
                       outline=targ_color,
                       width=3)
      text_string_to_show = f"{label_name} ({round(score.item(), 3)})"
      text_string_to_show = f"{label_name} ({round(score.item(), 3)})"
      draw.text(xy=(x,y),text=text_string_to_show,fill="white",font=font)

    del draw
    ### 5. Create logic for outputting information message ###

    # Setup set of target items to discover
    target_items = {"trash", "bin", "hand"}
    detected_items = set(detected_class_name_text_labels)

    # If no items detected or trash, bin, hand not in detected items, return notification
    if not detected_items & target_items:
        return_string = (
            f"No trash, bin or hand detected at confidence threshold {conf_threshold}. "
            "Try another image or lowering the confidence threshold."
        )
        print(return_string)
        return image, return_string

    # If there are missing items, say what the missing items are
    missing_items = target_items - detected_items
    if missing_items:
        return_string = (
            f"Detected the following items: {sorted(detected_items & target_items)}. But missing the following in order to get +1: {sorted(missing_items)}. "
            "If this is an error, try another image or altering the confidence threshold. "
            "Otherwise, the model may need to be updated with better data."
        )
        print(return_string)
        return image, return_string

    # If all target items are present (the final remaining case)
    return_string = f"+1! Found the following items: {sorted(detected_items)}, thank you for cleaning up the area!"
    print(return_string)
    return image, return_string

In [17]:
## final to write

%%writefile /content/demos/trashify_object_detector/app.py

# 1. Import the required libraries and packages
import gradio as gr
import torch
from PIL import Image, ImageDraw, ImageFont # could also use torch utilities for drawing

from transformers import AutoImageProcessor
from transformers import AutoModelForObjectDetection

### 2. Setup preprocessing and helper functions ###

# Setup target model path to load
# Note: Can load from Hugging Face or can load from local
model_save_path = "mrdbourke/rt_detrv2_finetuned_trashify_box_detector_v1"

# Load the model and preprocessor
# Because this app.py file is running directly on Hugging Face Spaces, the model will be loaded from the Hugging Face Hub
image_processor = AutoImageProcessor.from_pretrained(model_save_path)
model = AutoModelForObjectDetection.from_pretrained(model_save_path)

# Set the target device (use CUDA/GPU if it is available)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# Get the id2label dictionary from the model
id2label = model.config.id2label

# Set up a colour dictionary for plotting boxes with different colours
color_dict = {
    "bin": "green",
    "trash": "blue",
    "hand": "purple",
    "trash_arm": "yellow",
    "not_trash": "red",
    "not_bin": "red",
    "not_hand": "red",
}

### 3. Create function to predict on a given image with a given confidence threshold ###
def predict_on_image(image, conf_threshold):
    # Make sure model is in eval mode
    model.eval()

    # Make a prediction on target image
    with torch.no_grad():
        inputs = image_processor(images=[image], return_tensors="pt")
        model_outputs = model(**inputs.to(device))

        target_sizes = torch.tensor([[image.size[1], image.size[0]]]) # -> [batch_size, height, width]

        # Post process the raw outputs from the model
        results = image_processor.post_process_object_detection(model_outputs,
                                                                threshold=conf_threshold,
                                                                target_sizes=target_sizes)[0]

    # Return all items in results to CPU (we'll want this for displaying outputs with matplotlib)
    for key, value in results.items():
        try:
            results[key] = value.item().cpu() # can't get scalar as .item() so add try/except block
        except:
            results[key] = value.cpu()

    ### 4. Draw the predictions on the target image ###

    # Can return results as plotted on a PIL image (then display the image)
    draw = ImageDraw.Draw(image)

    # Get a font from ImageFont
    font = ImageFont.load_default(size=20)

    # Get class names as text for print out
    detected_class_name_text_labels = []

    # Iterate through the predictions of the model and draw them on the target image
    for box, score, label in zip(results["boxes"], results["scores"], results["labels"]):
        # Create coordinates
        x, y, x2, y2 = tuple(box.tolist())

        # Get label_name
        label_name = id2label[label.item()]
        targ_color = color_dict[label_name]
        detected_class_name_text_labels.append(label_name)

        # Draw the rectangle
        draw.rectangle(xy=(x, y, x2, y2),
                       outline=targ_color,
                       width=3)

        # Create a text string to display
        text_string_to_show = f"{label_name} ({round(score.item(), 3)})"

        # Draw the text on the image
        draw.text(xy=(x, y),
                  text=text_string_to_show,
                  fill="white",
                  font=font)

    # Remove the draw each time
    del draw

    ### 5. Create logic for outputting information message ###

    # Setup set of target items to discover
    target_items = {"trash", "bin", "hand"}
    detected_items = set(detected_class_name_text_labels)

    # If no items detected or trash, bin, hand not in detected items, return notification
    if not detected_items & target_items:
        return_string = (
            f"No trash, bin or hand detected at confidence threshold {conf_threshold}. "
            "Try another image or lowering the confidence threshold."
        )
        print(return_string)
        return image, return_string

    # If there are missing items, say what the missing items are
    missing_items = target_items - detected_items
    if missing_items:
        return_string = (
            f"Detected the following items: {sorted(detected_items & target_items)}. But missing the following in order to get +1: {sorted(missing_items)}. "
            "If this is an error, try another image or altering the confidence threshold. "
            "Otherwise, the model may need to be updated with better data."
        )
        print(return_string)
        return image, return_string

    # If all target items are present (the final remaining case)
    return_string = f"+1! Found the following items: {sorted(detected_items)}, thank you for cleaning up the area!"
    print(return_string)
    return image, return_string

### 6. Setup the demo application to take in image, make a prediction with our model, return the image with drawn predicitons ###

# Write description for our demo application
description = """
Help clean up your local area! Upload an image and get +1 if there is all of the following items detected: trash, bin, hand.

Model is a fine-tuned version of [RT-DETRv2](https://huggingface.co/docs/transformers/main/en/model_doc/rt_detr_v2#transformers.RTDetrV2Config) on the [Trashify dataset](https://huggingface.co/datasets/mrdbourke/trashify_manual_labelled_images).

See the full data loading and training code on [learnhuggingface.com](https://www.learnhuggingface.com/notebooks/hugging_face_object_detection_tutorial).

This version is v4 because the first three versions were using a different model and did not perform as well, see the [README](https://huggingface.co/spaces/mrdbourke/trashify_demo_v4/blob/main/README.md) for more.
"""

# Create the Gradio interface to accept an image and confidence threshold and return an image with drawn prediction boxes
demo = gr.Interface(
    fn=predict_on_image,
    inputs=[
        gr.Image(type="pil", label="Target Image"),
        gr.Slider(minimum=0, maximum=1, value=0.3, label="Confidence Threshold")
    ],
    outputs=[
        gr.Image(type="pil", label="Image Output"),
        gr.Text(label="Text Output")
    ],
    title="🚮 Trashify Object Detection Demo V4",
    description=description,
    # Examples come in the form of a list of lists, where each inner list contains elements to prefill the `inputs` parameter with
    # See where the examples originate from here: https://huggingface.co/datasets/mrdbourke/trashify_examples/
    examples=[
        ["trashify_examples/trashify_example_1.jpeg", 0.3],
        ["trashify_examples/trashify_example_2.jpeg", 0.3],
        ["trashify_examples/trashify_example_3.jpeg", 0.3],
    ],
    cache_examples=True
)

# Launch the demo
demo.launch()

Writing /content/demos/trashify_object_detector/app.py


## Making a requirements file

In [18]:
%%writefile /content/demos/trashify_object_detector/requirements.txt
timm
gradio
torch
transformers

Writing /content/demos/trashify_object_detector/requirements.txt


In [19]:
%%writefile /content/demos/trashify_object_detector/README.md
---
title: Trashify Demo V4 🚮
emoji: 🗑️
colorFrom: purple
colorTo: blue
sdk: gradio
sdk_version: 5.34.0
app_file: app.py
pinned: false
license: apache-2.0
---

# 🚮 Trashify Object Detector V4

Object detection demo to detect `trash`, `bin`, `hand`, `trash_arm`, `not_trash`, `not_bin`, `not_hand`.

Used as example for encouraging people to cleanup their local area.

If `trash`, `hand`, `bin` all detected = +1 point.

## Dataset

All Trashify models are trained on a custom hand-labelled dataset of people picking up trash and placing it in a bin.

The dataset can be found on Hugging Face as [`mrdbourke/trashify_manual_labelled_images`](https://huggingface.co/datasets/mrdbourke/trashify_manual_labelled_images).

## Demos

* [V1](RahulKate-173/rt_detrv2_finetuned_trashify_box_detector_v1) = Fine-tuned [Conditional DETR](https://huggingface.co/docs/transformers/en/model_doc/conditional_detr) model trained *without* data augmentation.
## Learn more

See the full end-to-end code of how this demo was built at [learnhuggingface.com](https://www.learnhuggingface.com/notebooks/hugging_face_object_detection_tutorial).

Writing /content/demos/trashify_object_detector/README.md


## Making an examples folder

In [20]:
# Make a directory to save examples to
from pathlib import Path

demo_example_dir = "/content/demos/trashify_object_detector/trashify_examples/"
Path(demo_example_dir).mkdir(exist_ok=True, parents=True)

In [21]:
# Download the examples from Hugging Face Datasets
from datasets import load_dataset

trashify_examples = load_dataset("mrdbourke/trashify_examples")
trashify_examples

README.md:   0%|          | 0.00/661 [00:00<?, ?B/s]

trashify_examples/trashify_example_1.jpe(…):   0%|          | 0.00/501k [00:00<?, ?B/s]

trashify_examples/trashify_example_2.jpe(…):   0%|          | 0.00/1.07M [00:00<?, ?B/s]

trashify_examples/trashify_example_3.jpe(…):   0%|          | 0.00/927k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image'],
        num_rows: 3
    })
})

In [22]:
for i, sample in enumerate(trashify_examples["train"]):
    save_path = Path(demo_example_dir, f"trashify_example_{i+1}.jpeg")
    print(f"[INFO] Saving image to: {save_path}")
    sample["image"].save(save_path)

[INFO] Saving image to: /content/demos/trashify_object_detector/trashify_examples/trashify_example_1.jpeg
[INFO] Saving image to: /content/demos/trashify_object_detector/trashify_examples/trashify_example_2.jpeg
[INFO] Saving image to: /content/demos/trashify_object_detector/trashify_examples/trashify_example_3.jpeg


In [23]:
!ls /content/demos/trashify_object_detector/

app.py	README.md  requirements.txt  trashify_examples


## Uploading our demo to Hugging Face Spaces

In [25]:
!pip install -q huggingface_hub
from huggingface_hub import notebook_login
notebook_login()

In [33]:
# 1. Import the required methods for uploading to the Hugging Face Hub
from huggingface_hub import (
    create_repo,
    get_full_repo_name,
    upload_file, # for uploading a single file (if necessary)
    upload_folder # for uploading multiple files (in a folder)
)

# 2. Define the parameters we'd like to use for the upload
LOCAL_DEMO_FOLDER_PATH_TO_UPLOAD = "/content/demos/trashify_object_detector"
HF_TARGET_SPACE_NAME = "trashify_demo_v4"
HF_REPO_TYPE = "space" # we're creating a Hugging Face Space
HF_SPACE_SDK = "gradio"
HF_TOKEN = "hf_*********************************", # optional: set to your Hugging Face token (but I'd advise storing this as an environment variable as previously discussed)

# 3. Create a Space repository on Hugging Face Hub
print(f"[INFO] Creating repo on Hugging Face Hub with name: {HF_TARGET_SPACE_NAME}")
create_repo(
    repo_id=HF_TARGET_SPACE_NAME,
    token="hf_*********************************", # optional: set token manually (though it will be automatically recognized if it's available as an environment variable)
    repo_type=HF_REPO_TYPE,
    private=False, # set to True if you don't want your Space to be accessible to others
    space_sdk=HF_SPACE_SDK,
    exist_ok=True, # set to False if you want an error to raise if the repo_id already exists
)

# 4. Get the full repository name (e.g. {username}/{model_id} or {username}/{space_name})
full_hf_repo_name = get_full_repo_name(model_id=HF_TARGET_SPACE_NAME,token=HF_TOKEN)
print(f"[INFO] Full Hugging Face Hub repo name: {full_hf_repo_name}")

# 5. Upload our demo folder
print(f"[INFO] Uploading {LOCAL_DEMO_FOLDER_PATH_TO_UPLOAD} to repo: {full_hf_repo_name}")
folder_upload_url = upload_folder(
    repo_id=full_hf_repo_name,
    folder_path=LOCAL_DEMO_FOLDER_PATH_TO_UPLOAD,
    path_in_repo=".", # upload our folder to the root directory ("." means "base" or "root", this is the default)
    token=HF_TOKEN, # optional: set token manually
    repo_type=HF_REPO_TYPE,
    commit_message="Uploading Trashify box detection model app.py"
)
print(f"[INFO] Demo folder successfully uploaded with commit URL: {folder_upload_url}")

[INFO] Creating repo on Hugging Face Hub with name: trashify_demo_v4
[INFO] Full Hugging Face Hub repo name: RahulKate-173/trashify_demo_v4
[INFO] Uploading /content/demos/trashify_object_detector to repo: RahulKate-173/trashify_demo_v4


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...s/trashify_example_2.jpeg: 100%|##########|  361kB /  361kB            

  ...s/trashify_example_3.jpeg: 100%|##########|  278kB /  278kB            

[INFO] Demo folder successfully uploaded with commit URL: https://huggingface.co/spaces/RahulKate-173/trashify_demo_v4/commit/4c16e247267c99025c09647dadab8b2fa5a34891


## Testing the hosted demo

In [35]:
from IPython.display import HTML

# You can get embeddable HTML code for your demo by clicking the "Embed" button on the demo page
HTML(data='''
<iframe
    src="https://rahulkate-173-trashify-demo-v4.hf.space"
    frameborder="0"
    width="850"
    height="1000"
></iframe>
''')